# Planners-1-Introduction-Csharp : la planification automatique from-scratch

**Navigation** : [<< Setup](Planners-0-Setup.ipynb) | [Twin Python unified-planning](Planners-1-Introduction.ipynb) | [PDDL Basics >>](Planners-2-PDDL-Basics.ipynb)

**Twin C#** du notebook Python [Planners-1-Introduction](Planners-1-Introduction.ipynb) (Python + lib `unified-planning`). Cette **premiere tranche = les fondations** : le modele **STRIPS** (Fikes & Nilsson, 1971), l'espace d'etats, et un **planificateur BFS from-scratch** qui resout le probleme canonique de l'interrupteur. Implementation **BCL .NET seule, 0 NuGet**.

## Complementarite (#3801 Prong B)

| Twin | Outil | Valeur |
|------|-------|--------|
| Python (unified-planning) | lib `up.shortcuts` (modelisation + solveur 1 appel) | resoudre vite des problemes PDDL reels |
| **This (.NET from-scratch)** | implementation C# main : STRIPS + BFS + state-space | **comprendre** la semantique STRIPS et la recherche dans l'espace d'etats |

Pas d'equivalent NuGet maintenu et didactique a `unified-planning` en .NET → from-scratch = la valeur pedagogique. On code chaque brique (predicat, operateur, transition, recherche) pour comprendre **comment** un planificateur fonctionne, pas seulement comment l'invoquer.

**Stack technique** : BCL .NET seule, **0 NuGet**. Kernel `.net-csharp` (.NET Interactive).


## 1. Qu'est-ce que la planification ?

La **planification automatique** est la branche de l'IA qui genere des **sequences d'actions** pour atteindre un objectif. Etant donne :
- un **etat initial** (le monde tel qu'il est),
- un ensemble d'**actions** (chacune decrite par ses **preconditions** et ses **effets**),
- un **but** (un etat a atteindre),

le planificateur cherche automatiquement une suite d'actions qui mene de l'etat initial au but. La question n'est pas « que predire ? » (apprentissage) mais « **que faire ?** ».

**Contraste** :
- **Recherche (Search series)** : on explore un graphe donne. La planification **construit** ce graphe (l'espace d'etats) a partir de la semantique des actions.
- **CSP (Constraint series)** : on satisfait des contraintes sur des variables. La planification cherche une **sequence** (l'ordre compte).

La planification est une technologie eprouvee : elle pilote des robots, optimise la logistique, et a dirige des engins spatiaux autonomes (Remote Agent sur Deep Space 1).


## 2. Le modele STRIPS

**STRIPS** (STanford Research Institute Problem Solver, Fikes & Nilsson 1971) est le formalisme canonique. L'etat du monde = un **ensemble de predicats** (faits vrais). Une action (appelee **operateur**) comporte :

- un **nom** (ex: `allumer(lampe)`),
- des **preconditions** : predicats qui doivent etre vrais pour que l'action soit applicable,
- des **effets** : predicats **ajoutes** (deviennent vrais) et **retires** (deviennent faux) apres l'action.

La **transition** : si l'etat courant contient toutes les preconditions, l'action est applicable. L'etat resultant = (etat courant − effets de retrait) ∪ effets d'ajout.

### Formalisme

Soit un operateur $o = (pre(o), add(o), del(o))$. Pour un etat $S$ :

- **applicable** ssi $pre(o) \subseteq S$
- **successeur** : $S' = (S \setminus del(o)) \cup add(o)$

Un **plan** = sequence d'operateurs $o_1, o_2, \dots, o_n$ telle que chaque $o_i$ est applicable dans l'etat atteint apres $o_1, \dots, o_{i-1}$, et le but est atteint a la fin.


## 3. Setup et helpers invariant

Helper de formatage invariant (evite la collision virgule-decimale FR dans les sorties).


In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

static string FI(double x, string fmt = "F4") => x.ToString(fmt, CultureInfo.InvariantCulture);

Console.WriteLine("Setup OK — kernel .net-csharp, BCL seule.");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Setup OK — kernel .net-csharp, BCL seule.


## 4. Implementation du modele STRIPS from-scratch

On represente l'**etat** comme un `HashSet<string>` de predicats (ex: `{"lampe_allumee", "lampe_fonctionne"}`). Un **operateur** est une classe avec preconditions, ajouts, retraits. La transition applique l'operateur si applicable.


In [2]:
// Modele STRIPS from-scratch (BCL .NET seule).
// Un etat = HashSet<string> (ensemble de predicats vrais).

// Un operateur STRIPS : nom + preconditions + effets (add/del).
class Operator {
    public string Name { get; init; } = "";
    public HashSet<string> Pre { get; init; } = new();
    public HashSet<string> Add { get; init; } = new();
    public HashSet<string> Del { get; init; } = new();
}

// Une transition : applicable ssi pre <= S ; successeur = (S - del) | add.
static bool IsApplicable(Operator op, HashSet<string> s) => op.Pre.IsSubsetOf(s);

static HashSet<string> Apply(Operator op, HashSet<string> s) {
    // On ne devrait appeler Apply que si IsApplicable.
    var s2 = new HashSet<string>(s);
    foreach (var d in op.Del) s2.Remove(d);
    foreach (var a in op.Add) s2.Add(a);
    return s2;
}

// Le but est atteint ssi tous les predicats du but sont dans l'etat.
static bool GoalReached(HashSet<string> goal, HashSet<string> s) => goal.IsSubsetOf(s);

Console.WriteLine("Modele STRIPS compile : Operator + IsApplicable + Apply + GoalReached.");


Modele STRIPS compile : Operator + IsApplicable + Apply + GoalReached.


## 5. Exemple simple — l'interrupteur

Le probleme canonique de l'interrupteur (cf twin Python) :
- **Etat initial** : la lampe est eteinte.
- **Actions** : `allumer` (pre: lampe en bon etat ; add: lampe allumee), `eteindre` (add: lampe eteinte, del: lampe allumee).
- **But** : la lampe est allumee.

Ce probleme trivial sert de **sanity check** : un planificateur BFS doit trouver le plan `[allumer]`.


In [3]:
// Exemple de l'interrupteur (sanity check).

var allumer = new Operator {
    Name = "allumer",
    Pre = new HashSet<string> { "lampe_fonctionne" },
    Add = new HashSet<string> { "lampe_allumee" },
    Del = new HashSet<string> { "lampe_eteinte" }
};
var eteindre = new Operator {
    Name = "eteindre",
    Pre = new HashSet<string> { "lampe_allumee" },
    Add = new HashSet<string> { "lampe_eteinte" },
    Del = new HashSet<string> { "lampe_allumee" }
};

HashSet<string> initial = new HashSet<string> { "lampe_eteinte", "lampe_fonctionne" };
var goal = new HashSet<string> { "lampe_allumee" };

Console.WriteLine("Etat initial  : {" + string.Join(", ", initial.OrderBy(x => x)) + "}");
Console.WriteLine("But           : {" + string.Join(", ", goal) + "}");
Console.WriteLine("But atteint directement ? " + GoalReached(goal, initial));
Console.WriteLine("'allumer' applicable ? " + IsApplicable(allumer, initial));
var apres = Apply(allumer, initial);
Console.WriteLine("Apres 'allumer' : {" + string.Join(", ", apres.OrderBy(x => x)) + "}");
Console.WriteLine("But atteint apres 'allumer' ? " + GoalReached(goal, apres));


Etat initial  : {lampe_eteinte, lampe_fonctionne}


But           : {lampe_allumee}


But atteint directement ? False


'allumer' applicable ? True


Apres 'allumer' : {lampe_allumee, lampe_fonctionne}


But atteint apres 'allumer' ? True


### Interpretation — l'interrupteur

L'etat initial ne contient pas le but (`lampe_allumee` absent). L'action `allumer` est applicable car sa precondition `lampe_fonctionne` est satisfaite. Apres application, `lampe_allumee` est ajoutee → le but est atteint. Sanity OK : le plan `[allumer]` resout le probleme.

C'est exactement ce que ferait `unified-planning` (twin Python) en un appel — mais ici on **voit** chaque etape (precondition, ajout, retrait).


## 6. Le planificateur BFS from-scratch

Un planificateur = une **recherche dans l'espace d'etats**. L'espace d'etats = graphe ou les noeuds sont les etats et les aretes sont les operateurs applicables. On cherche un chemin de l'etat initial a un etat-but.

**BFS (Breadth-First Search)** = parcours en largeur. Garantit le **plan le plus court** (en nombre d'actions). Pour un petit espace d'etats, BFS suffit. Pour de grands espaces, on aurait besoin d'heuristiques (A*, relaxations — cf Planners-5).

Implementation : une file d'etats a explorer, un dictionnaire `etat -> (operateur, etat parent)` pour reconstruire le plan une fois le but atteint.


In [4]:
// Planificateur BFS from-scratch — trouve le plan le plus court.

static List<string>? PlanBFS(HashSet<string> initial, HashSet<string> goal, List<Operator> ops, int maxExpansions = 100000) {
    var queue = new Queue<HashSet<string>>();
    queue.Enqueue(initial);
    // came_from[etat] = (operateur applique, etat parent)
    var cameFrom = new Dictionary<string, (string opName, HashSet<string> parent)>(new StateComparer());
    cameFrom[StateKey(initial)] = ("__INIT__", initial);

    int expansions = 0;
    while (queue.Count > 0 && expansions < maxExpansions) {
        var current = queue.Dequeue();
        expansions++;
        if (GoalReached(goal, current)) {
            // Reconstruire le plan.
            var plan = new List<string>();
            var key = StateKey(current);
            while (cameFrom[key].opName != "__INIT__") {
                var (opName, parent) = cameFrom[key];
                plan.Add(opName);
                key = StateKey(parent);
            }
            plan.Reverse();
            return plan;
        }
        foreach (var op in ops) {
            if (IsApplicable(op, current)) {
                var next = Apply(op, current);
                var nkey = StateKey(next);
                if (!cameFrom.ContainsKey(nkey)) {
                    cameFrom[nkey] = (op.Name, current);
                    queue.Enqueue(next);
                }
            }
        }
    }
    return null;  // pas de plan trouve dans maxExpansions
}

// Cle string canonique pour un etat (predicats tries).
static string StateKey(HashSet<string> s) => string.Join("|", s.OrderBy(x => x));

// Comparateur d'etats base sur la cle string.
class StateComparer : IEqualityComparer<string> {
    public bool Equals(string? a, string? b) => a == b;
    public int GetHashCode(string s) => s?.GetHashCode() ?? 0;
}

Console.WriteLine("Planificateur BFS compile.");


Planificateur BFS compile.



(3,20): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(45,30): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(45,41): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Plan sur l'interrupteur

Lançons BFS sur le probleme de l'interrupteur. Il doit trouver `[allumer]` (1 action, le plus court).


In [5]:
// Plan sur l'interrupteur.
var ops = new List<Operator> { allumer, eteindre };
var plan = PlanBFS(initial, goal, ops);
if (plan != null) {
    Console.WriteLine($"Plan trouve ({plan.Count} action(s)) : [{string.Join(" -> ", plan)}]");
} else {
    Console.WriteLine("Aucun plan trouve.");
}


Plan trouve (1 action(s)) : [allumer]


## 7. Probleme plus riche — multi-interrupteurs (non-trivial)

L'interrupteur seul est trivial (1 action). Pour **mettre en valeur** le planificateur (cf #3801 Prong B — probleme non-trivial), considerons **N interrupteurs** a allumer dans un ordre quelconque, mais avec une contrainte : une **maitresse** (master switch) doit etre allumee avant de pouvoir allumer les autres (precondition chainee). Le but = tous les interrupteurs allumes.

Ce probleme a une **structure** (la maitresse d'abord) que BFS doit decouvrir — ce n'est pas un graphe a cout uniforme trivial.


In [6]:
// N interrupteurs + maitresse (probleme non-trivial).
int N = 3;
var multiOps = new List<Operator>();
// allumer_master : pre = rien ; add = master_on
multiOps.Add(new Operator {
    Name = "allumer_master",
    Pre = new HashSet<string>(),
    Add = new HashSet<string> { "master_on" },
    Del = new HashSet<string> { "master_off" }
});
// allumer_i : pre = master_on ; add = switch_i_on ; del = switch_i_off
for (int i = 1; i <= N; i++) {
    int idx = i;
    multiOps.Add(new Operator {
        Name = $"allumer_sw{idx}",
        Pre = new HashSet<string> { "master_on" },
        Add = new HashSet<string> { $"switch_{idx}_on" },
        Del = new HashSet<string> { $"switch_{idx}_off" }
    });
}

HashSet<string> multiInitial = new HashSet<string>();
multiInitial.Add("master_off");
for (int i = 1; i <= N; i++) multiInitial.Add($"switch_{i}_off");

var multiGoal = new HashSet<string> { "master_on" };
for (int i = 1; i <= N; i++) multiGoal.Add($"switch_{i}_on");

Console.WriteLine($"Probleme : {N} interrupteurs + 1 maitresse");
Console.WriteLine($"Etat initial : {multiInitial.Count} predicats (tout eteint)");
Console.WriteLine($"But : master_on + {N} switch_*_on");
Console.WriteLine();

var multiPlan = PlanBFS(multiInitial, multiGoal, multiOps);
if (multiPlan != null) {
    Console.WriteLine($"Plan trouve ({multiPlan.Count} actions) :");
    Console.WriteLine($"  [{string.Join(" -> ", multiPlan)}]");
    Console.WriteLine();
    // Verifier : master_on doit preceder tous allumer_sw*
    int masterIdx = multiPlan.IndexOf("allumer_master");
    Console.WriteLine($"'allumer_master' a la position {masterIdx + 1}/{multiPlan.Count} — doit etre PREMIERE (precondition chainee).");
    Console.WriteLine($"Structure decouverte par BFS : maitresse d'abord, puis les {N} interrupteurs (ordre arbitraire).");
} else {
    Console.WriteLine("Aucun plan trouve.");
}


Probleme : 3 interrupteurs + 1 maitresse


Etat initial : 4 predicats (tout eteint)


But : master_on + 3 switch_*_on


Plan trouve (4 actions) :


  [allumer_master -> allumer_sw1 -> allumer_sw2 -> allumer_sw3]


'allumer_master' a la position 1/4 — doit etre PREMIERE (precondition chainee).


Structure decouverte par BFS : maitresse d'abord, puis les 3 interrupteurs (ordre arbitraire).


### Interpretation — multi-interrupteurs

BFS decouvre automatiquement la **structure du probleme** :
- Il faut **d'abord** allumer la maitresse (seule action applicable sans precondition).
- **Puis** allumer chaque interrupteur (precondition `master_on` satisfaite).

Le plan a `N+1` actions : `allumer_master` puis `allumer_sw1`, `allumer_sw2`, ..., `allumer_swN` (l'ordre des `sw*` est arbitraire — BFS trouve l'un des plans optimaux). C'est exactement le genre de plan que `unified-planning` produirait, mais ici on **voit** la recherche explorer l'espace d'etats.

**Pourquoi c'est non-trivial** : sans precondition chainee, le probleme serait degenere (toutes les actions independantes). La maitresse force un **ordre partiel** que le planificateur doit decouvrir — c'est ce qui met BFS en valeur (cf #3801 Prong B, probleme non-degenere).


## 8. Explosion combinatoire de l'espace d'etats

Avec $n$ predicats (faits booléens), il y a jusqu'a $2^n$ etats possibles. Pour 10 predicats = 1024 etats, pour 20 = 1 million, pour 50 = $10^{15}$... C'est le **fleau de la combinatoire** qui motive les heuristiques (A*, relaxations — cf Planners-5).


In [7]:
// Explosion combinatoire : 2^n etats pour n predicats.
Console.WriteLine("Explosion combinatoire de l'espace d'etats (2^n) :");
Console.WriteLine();
Console.WriteLine("  n predicats | etats possibles");
Console.WriteLine("  ----------- | ---------------");
foreach (int n in new[] { 5, 10, 15, 20, 30, 50 }) {
    double states = Math.Pow(2, n);
    string repr = states < 1e6 ? FI(states, "F0") : states.ToString("E2", CultureInfo.InvariantCulture);
    Console.WriteLine($"  {n,11} | {repr,15}");
}
Console.WriteLine();
Console.WriteLine(">>> 50 predicats = ~1e15 etats : aucun planificateur aveugle (BFS) ne peut explorer.");
Console.WriteLine(">>> D'ou les HEURISTIQUES (A*, relaxations) pour guider la recherche (cf Planners-5).");


Explosion combinatoire de l'espace d'etats (2^n) :


  n predicats | etats possibles


  ----------- | ---------------


            5 |              32


           10 |            1024


...

           20 |       1.05E+006


           30 |       1.07E+009


           50 |       1.13E+015


>>> 50 predicats = ~1e15 etats : aucun planificateur aveugle (BFS) ne peut explorer.


>>> D'ou les HEURISTIQUES (A*, relaxations) pour guider la recherche (cf Planners-5).


## 9. Types de planification

La planification se declinent en plusieurs variantes (cf twin Python) :

| Type | Caractéristique | Exemple |
|------|-----------------|---------|
| **Classique** | etats discrets, actions instantanées, deterministes | empiler des blocs, logistique |
| **Temporelle** | actions avec durée, chevauchement | ordonnancement de tâches |
| **HTN** (Hierarchical Task Network) | décomposition de tâches abstraites | planification militaire, cuisine |
| **Neuro-symbolique** | LLM + planificateur formel | agent LLM avec vérification |

Ce notebook couvre les **fondations classiques** (STRIPS + BFS). Les notebooks suivants (PDDL, heuristiques, etc.) approfondissent.


## 10. Exercices

Trois exercices pour aller plus loin. Chaque stub est conforme a la regle C.1 (pas d'erreur volontaire — le notebook s'execute de bout en bout meme exercices non completes).


### Exercice 1 — Le monde des blocs (blocks world)

**Enonce** : Modelisez le **monde des blocs** : 3 blocs A, B, C. Etat initial : A sur la table, B sur A, C sur la table. But : C sur B sur A (pyramide). Actions : `deplacer(x, y, z)` (pre: x sur y, bras libre ; add: x sur z ; del: x sur y).

**Etape 1** : Definissez les operateurs (avec les predicats `sur(x,y)`, ` Bras_libre`, `sur_table(x)`).
**Etape 2** : Codez l'etat initial et le but.
**Etape 3** : Lancez `PlanBFS` et observez le plan.

**Indice** : attention aux precondition `Bras_libre` — un seul bloc peut etre deplace a la fois. C'est un probleme classique (Sussman anomaly) plus riche que l'interrupteur.


In [8]:
// Exercice 1 — Monde des blocs (A sur table, B sur A, C sur table -> C sur B sur A).
// TODO etudiant : definissez les operateurs deplacer + etat initial + but, lancez PlanBFS.
Console.WriteLine("Exercice 1 a completer : monde des blocs (blocks world).");


Exercice 1 a completer : monde des blocs (blocks world).


### Exercice 2 — Ajouter une heuristique (A* au lieu de BFS)

**Enonce** : BFS trouve le plan le plus court mais explore en aveugle. Implementez **A*** avec une heuristique simple : nombre de predicats du but non encore satisfaits dans l'etat courant.

**Questions** : (a) A* explore-t-il moins d'etats que BFS sur le multi-interrupteurs ? (b) L'heuristique est-elle **admissible** (ne surestime jamais le cout) ?

**Indice** : remplacez la `Queue` BFS par une `PriorityQueue` ordonnée par `g + h` (cout parcouru + heuristique). `g` = nombre d'actions depuis l'initial, `h` = predicats du but manquants.


In [9]:
// Exercice 2 — A* avec heuristique (predicats du but manquants).
// TODO etudiant : PriorityQueue g+h, h = |goal - S|, comparez explorations vs BFS.
Console.WriteLine("Exercice 2 a completer : A* avec heuristique de but non-satisfait.");


Exercice 2 a completer : A* avec heuristique de but non-satisfait.


### Exercice 3 — Echec : probleme non-realisable

**Enonce** : Creez un probleme **sans solution** (ex: but = lampe allumee, mais `lampe_fonctionne` absent de l'etat initial et aucune action ne l'etablit). Observez le comportement de `PlanBFS` (retourne `null` apres `maxExpansions`).

**Questions** : (a) Combien d'etats BFS explore-t-il avant d'abandonner ? (b) Comment detecter l'**insatisfiabilite** plus tot (ex: analyse des preconditions — un fait du but non atteignable par aucun effet d'ajout) ?

**Indice** : `maxExpansions` borne l'exploration. Un fait du but qui n'apparait dans aucun `Add` d'operateur est **mort** — detection statique possible.


In [10]:
// Exercice 3 — Probleme non-realisable (but avec fait non-atteignable).
// TODO etudiant : etat initial sans lampe_fonctionne, but lampe_allumee, observez null.
Console.WriteLine("Exercice 3 a completer : probleme non-realisable + detection statique.");


Exercice 3 a completer : probleme non-realisable + detection statique.


## Conclusion — Tranche 1 fondations

**Ce que nous avons appris** :

| Concept | Definition |
|---------|------------|
| **Planification** | Generer une sequence d'actions pour atteindre un but |
| **STRIPS** | Formalisme : etat = predicats, operateur = pre/add/del |
| **Transition** | $(S \setminus del) \cup add$ si $pre \subseteq S$ |
| **BFS planner** | Recherche en largeur dans l'espace d'etats (plan le plus court) |
| **Explosion combinatoire** | $2^n$ etats pour $n$ predicats → motive les heuristiques |

**Lecon cles** : un planificateur = une **recherche dans l'espace d'etats construit a partir de la semantique des actions**. Le twin Python `unified-planning` invoque un solveur ; ce twin from-scratch **construit** le solveur (modele STRIPS + BFS). Les deux sont complementaires : comprendre vs resoudre vite.

**Suite** : PDDL (Planners-2 — le langage standard pour decrire ces problemes), heuristiques (Planners-5 — A*, relaxations pour combattre l'explosion combinatoire).

---

*Implementation from-scratch, BCL .NET seule, 0 NuGet. Twin du notebook Python unified-planning [Planners-1-Introduction](Planners-1-Introduction.ipynb). Marathon #4956 (premier jumeau C# de la serie Planners).*
